# 12 — Build the gold annotation kit

Samples sentences from the held-out NER test split and produces one Excel
workbook per annotator, plus the answer key needed later for scoring.

Three deliberate choices:

- **Blind.** Annotators never see the projection labels. Showing them would
  anchor judgments toward the projection, and agreement with the projection is
  exactly what we want to measure.
- **Whole-token spans.** `Aizawlah` is tagged in full. Annotators should not
  have to reason about morpheme boundaries, and it matches what a tagger
  predicts.
- **Sampled from the test split.** Those sentences were never seen in training,
  so the result serves as a clean evaluation as well as a projection audit.

The workbook carries four sheets: instructions, a decision flowchart, worked
examples drawn from the corpus, and the annotation grid.

**Run from the repository root.** Kernel: `Python (tka)`.

> **Cell 4 needs you.** It selects candidate example sentences using projection
> labels, which are only about 87% reliable. Check each one and correct it
> before the workbooks are built in Cell 6.

## Cell 1: Setup

In [6]:
from pathlib import Path
import json, sys, random
from collections import Counter, defaultdict

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

BIO = ROOT / "data" / "processed" / "bio_v2"
OUT = ROOT / "annotation"
OUT.mkdir(exist_ok=True)

src = BIO / "mizo_ner_test.json"
print(("  ok   " if src.exists() else "  MISS ") + str(src.relative_to(ROOT)))
if not src.exists():
    sys.exit("Run 02b first")

# ---- settings ------------------------------------------------------------
N_SENTENCES = 300           # 300 is about 13 h per annotator; 500 about 22 h
ANNOTATORS  = ["A", "B"]    # both annotate everything
SEED        = 20260101
# --------------------------------------------------------------------------
print(f"\n{N_SENTENCES} sentences for {len(ANNOTATORS)} annotators")

Repo root: C:\Users\Haulai\mizo-ner
  ok   data\processed\bio_v2\mizo_ner_test.json

300 sentences for 2 annotators


## Cell 2: Sample

In [7]:
records = json.load(open(src, encoding="utf-8"))
print(f"test split: {len(records):,} sentences")

rng = random.Random(SEED)
buckets = defaultdict(list)
for r in records:
    n = len(r["tokens"])
    buckets["short" if n <= 8 else ("medium" if n <= 15 else "long")].append(r)

quota = {"short": int(N_SENTENCES*0.3), "medium": int(N_SENTENCES*0.45)}
quota["long"] = N_SENTENCES - quota["short"] - quota["medium"]

sample, chosen = [], set()
for k, q in quota.items():
    take = rng.sample(buckets[k], min(q, len(buckets[k])))
    sample.extend(take); chosen.update(id(x) for x in take)
    print(f"  {k:<7}{len(take):>4}")
rng.shuffle(sample)

print(f"\n{len(sample)} sentences, {sum(len(r['tokens']) for r in sample):,} tokens")
proj = Counter(t[2:] for r in sample for t in r["tags"] if t.startswith("B-"))
print("projection marks:", dict(proj.most_common()))

key = [{"key_id": i+1, "corpus_id": r["id"], "tokens": r["tokens"],
        "projection_tags": r["tags"]} for i, r in enumerate(sample)]
with open(OUT / "gold_sample_key.json", "w", encoding="utf-8") as f:
    json.dump(key, f, ensure_ascii=False, indent=1)
print(f"\n-> annotation/gold_sample_key.json   (do NOT send to annotators)")

test split: 44,118 sentences
  short    90
  medium  135
  long     75

300 sentences, 3,644 tokens
projection marks: {'PERSON': 224, 'GPE': 75, 'ORG': 64, 'NORP': 17, 'WORK_OF_ART': 5, 'LOC': 5, 'FAC': 5, 'PRODUCT': 2, 'LANGUAGE': 2, 'EVENT': 1}

-> annotation/gold_sample_key.json   (do NOT send to annotators)


## Cell 3: Find candidate worked examples

Searches the corpus (outside the annotation sample, so no answers are leaked)
for sentences illustrating the distinctions most likely to cause disagreement.

In [8]:
pool = [r for r in records if id(r) not in chosen]
print(f"searching {len(pool):,} sentences outside the sample")

def spans(r):
    out, i, tk, tg = [], 0, r["tokens"], r["tags"]
    while i < len(tk):
        if tg[i].startswith("B-"):
            lab = tg[i][2:]; j = i+1
            while j < len(tk) and tg[j] == f"I-{lab}": j += 1
            out.append((" ".join(tk[i:j]), lab)); i = j
        else:
            i += 1
    return out

SUFFIX = ("ah", "-an", "in", "-a")
WANTED = [
 ("suffixed place name",     lambda r,s: any(l=="GPE" and t.endswith(SUFFIX) for t,l in s)),
 ("person with a title",     lambda r,s: any(l=="PERSON" and t.split()[0] in
                                             ("Pu","Pi","Dr.","Dr","Rev.","Rev","Upa") for t,l in s)),
 ("multi-word person",       lambda r,s: any(l=="PERSON" and len(t.split())>=2 for t,l in s)),
 ("NORP",                    lambda r,s: any(l=="NORP" for t,l in s)),
 ("LANGUAGE",                lambda r,s: any(l=="LANGUAGE" for t,l in s)),
 ("organisation",            lambda r,s: any(l=="ORG" for t,l in s)),
 ("facility",                lambda r,s: any(l=="FAC" for t,l in s)),
 ("non-political location",  lambda r,s: any(l=="LOC" for t,l in s)),
 ("two types in one clause", lambda r,s: len({l for _,l in s})>=2),
 ("work of art",             lambda r,s: any(l=="WORK_OF_ART" for t,l in s)),
 ("no entity at all",        lambda r,s: len(s)==0),
]

cands = defaultdict(list)
for r in pool:
    s = spans(r)
    n = len(r["tokens"])
    if not (5 <= n <= 16):
        continue
    for name, test in WANTED:
        if len(cands[name]) < 4 and test(r, s):
            cands[name].append((r, s))

print()
for name, _ in WANTED:
    print(f"  {name:<26}{len(cands[name])} candidates")

searching 43,818 sentences outside the sample

  suffixed place name       4 candidates
  person with a title       4 candidates
  multi-word person         4 candidates
  NORP                      4 candidates
  LANGUAGE                  4 candidates
  organisation              4 candidates
  facility                  4 candidates
  non-political location    4 candidates
  two types in one clause   4 candidates
  work of art               4 candidates
  no entity at all          0 candidates


## Cell 4: Review the candidates — **your input needed**

Each block below prints a sentence with the projection's labels. The projection
is roughly 87% reliable, so some are wrong. Read each, and note the ones you
want to keep and any label that needs correcting.

Edit `EXAMPLES` in the next cell accordingly.

In [9]:
print("=" * 74)
for name, _ in WANTED:
    if not cands[name]:
        continue
    print(f"\n### {name.upper()}")
    for k, (r, s) in enumerate(cands[name][:2]):
        print(f"\n  [{name} #{k}]  corpus_id {r['id']}")
        print(f"  {' '.join(r['tokens'])}")
        if s:
            for t, l in s:
                print(f"      {t:<28} {l}")
        else:
            print("      (projection found no entity)")
print("\n" + "=" * 74)
print("Copy the ones you want into EXAMPLES in Cell 5, fixing any wrong label.")


### SUFFIXED PLACE NAME

  [suffixed place name #0]  corpus_id 240
  Hemi avang hian Mizoramah mi 9,653 raltlan an awm mek.
      Mizoramah                    GPE

  [suffixed place name #1]  corpus_id 278
  Dama-an Mary-i Boston-a a kalpui chu pawi ka ti lo.
      Dama-an                      PERSON
      Mary-i                       PERSON
      Boston-a                     GPE

### PERSON WITH A TITLE

  [person with a title #0]  corpus_id 1272
  Tlangmawii pa, Pu Thangkhuma a thih leh tak avangin, ka insawn.
      Tlangmawii                   ORG
      Pu Thangkhuma                PERSON

  [person with a title #1]  corpus_id 6081
  Pu Saitluanga, Director, Tourism Department chuan lawmthu a sawi bawk.
      Pu Saitluanga,               PERSON
      Tourism Department           ORG

### MULTI-WORD PERSON

  [multi-word person #0]  corpus_id 25
  Armed Veng hian player langsar an nei hmasa zawk a;
      Armed Veng                   PERSON

  [multi-word person #1]  corpus_id 92
  A

## Cell 5: The worked examples — **edit this cell**

Replace the placeholder list with your chosen sentences. Format:

```python
("Mizo sentence here", [("entity text", "TYPE"), ...], "one-line reason")
```

Use `[]` for a sentence with no entities. Keep the reason short; it is what the
annotator reads.

In [10]:
# EDIT ME. Start from the Cell 4 output and correct any label that is wrong.
EXAMPLES = [
 ("Aizawlah an thuthmun tur ngaihtuah mek a ni.",
  [("Aizawlah", "GPE")],
  "Tag the whole word including the -ah suffix. Aizawl has a government, so GPE."),

 ("Pu Lalthanhawla chuan thu a sawi.",
  [("Pu Lalthanhawla", "PERSON")],
  "The title Pu is part of the name. Tag Pu as B-PERSON and the name as I-PERSON."),

 ("Mizo tawng hi Tibeto-Burman ṭawng a ni.",
  [("Mizo tawng", "LANGUAGE")],
  "Here Mizo means the language, so LANGUAGE, not NORP."),

 ("Mizo te chu Zoram-ah an cheng.",
  [("Mizo", "NORP"), ("Zoram-ah", "GPE")],
  "Here Mizo means the people, so NORP. Compare the previous example."),

 ("Mizoram sorkar chuan hnathawh thar a puang.",
  [("Mizoram sorkar", "ORG")],
  "A place name used as a governing body is ORG, not GPE."),

 ("Khuai leh ramsa te chu ngaihtuah tur a ni.",
  [],
  "khuai (bees) and ramsa (wild animals) are common nouns, not names. All O."),

 # add more from Cell 4 here
]

for text, ents, why in EXAMPLES:
    toks = text.split()
    for e, _ in ents:
        assert all(w in toks for w in e.split()), f"'{e}' not found in: {text}"
print(f"{len(EXAMPLES)} worked examples, all entity strings verified present")

6 worked examples, all entity strings verified present


## Cell 6: Build the workbooks

In [11]:
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Font, PatternFill, Alignment

TYPES = ["PERSON","GPE","ORG","NORP","LOC","LANGUAGE",
         "WORK_OF_ART","FAC","PRODUCT","EVENT","LAW"]
TAGS  = ["O"] + [f"{p}-{e}" for e in TYPES for p in ("B","I")]

GLOSS = {
 "PERSON":"People, real or fictional. Titles (Pu, Pi, Dr., Rev.) included when part of the name.",
 "GPE":"Countries, states, districts, cities, villages - places with a government.",
 "ORG":"Companies, institutions, churches, government bodies, parties, clubs, committees.",
 "NORP":"Nationalities, ethnic groups, religious groups, political affiliations.",
 "LOC":"Places without a government: rivers, hills, regions, roads, valleys.",
 "LANGUAGE":"Names of languages.",
 "WORK_OF_ART":"Titles of books, songs, films, paintings, newspapers.",
 "FAC":"Buildings, airports, bridges, hospitals, schools, halls, highways.",
 "PRODUCT":"Manufactured or branded objects.",
 "EVENT":"Named events: festivals, wars, tournaments, conferences.",
 "LAW":"Named laws, acts, treaties, court cases.",
}

FLOW = [
 ("STEP 1", "Is this word part of a NAME?", ""),
 ("", "No  -> tag O. Common nouns are O even when they seem important.", ""),
 ("", "Yes -> go to Step 2.", ""),
 ("", "", ""),
 ("STEP 2", "What kind of name is it?", ""),
 ("", "A human being", "-> PERSON (include Pu, Pi, Dr. if attached)"),
 ("", "A group of people", "-> NORP (Mizo, Christian, Congress supporter)"),
 ("", "A language", "-> LANGUAGE"),
 ("", "A place", "-> go to Step 3"),
 ("", "A body of people that acts", "-> ORG"),
 ("", "A building or structure", "-> FAC"),
 ("", "A book, song, film, paper", "-> WORK_OF_ART"),
 ("", "A named event", "-> EVENT"),
 ("", "A branded object", "-> PRODUCT"),
 ("", "A named law or act", "-> LAW"),
 ("", "", ""),
 ("STEP 3", "The name is a place. Does it have a government?", ""),
 ("", "Yes - country, state, district, city, village", "-> GPE"),
 ("", "No - river, hill, region, road, valley", "-> LOC"),
 ("", "It is acting as a government or body here", "-> ORG"),
 ("", "", ""),
 ("STEP 4", "First word of the entity -> B-TYPE. Later words -> I-TYPE.", ""),
 ("", "Tag the WHOLE word, suffix included: Aizawlah -> B-GPE.", ""),
]

HDR  = PatternFill("solid", fgColor="DDDDDD")
SENT = PatternFill("solid", fgColor="F2F7FF")
GOOD = PatternFill("solid", fgColor="EAF6EA")

def build(annotator, path):
    wb = Workbook()

    ws = wb.active; ws.title = "Instructions"
    for i, (t, b) in enumerate([
        ("Mizo Named Entity Annotation", True), ("", False),
        (f"Annotator: {annotator}", False), ("", False),
        ("WHAT TO DO", True),
        ("Open the 'Annotate' sheet. Each row is one word. Choose a tag in the", False),
        ("Tag column from the dropdown. Most words are O.", False), ("", False),
        ("HOW TAGS WORK", True),
        ("B-TYPE  first word of an entity", False),
        ("I-TYPE  each following word of the SAME entity", False),
        ("O       not part of any entity", False), ("", False),
        ("Pu / B-PERSON   Lalthanhawla / I-PERSON   chuan / O   Aizawlah / B-GPE", False),
        ("", False),
        ("BEFORE YOU START", True),
        ("Read the 'Decision guide' sheet, then the 'Examples' sheet.", False),
        ("The examples are real sentences from the corpus with correct answers.", False),
        ("", False),
        ("IF YOU ARE UNSURE", True),
        ("Put your best guess in Tag and explain in the Note column.", False),
        ("Disagreements are discussed afterwards - that is expected and useful.", False),
        ("Please work independently. Do not consult the other annotator.", False),
        ("Do not delete, insert or reorder rows.", False),
    ], start=1):
        c = ws.cell(row=i, column=1, value=t)
        if b: c.font = Font(bold=True, size=12)
    ws.column_dimensions["A"].width = 92

    wf = wb.create_sheet("Decision guide")
    wf.append(["", "Question / rule", "Tag"])
    for c in wf[1]: c.font = Font(bold=True); c.fill = HDR
    for a, b, c_ in FLOW:
        wf.append([a, b, c_])
        if a.startswith("STEP"):
            for cell in wf[wf.max_row]: cell.font = Font(bold=True)
    for col, w in (("A", 10), ("B", 58), ("C", 40)):
        wf.column_dimensions[col].width = w

    we = wb.create_sheet("Examples")
    we.append(["#", "Sentence", "Word", "Correct tag", "Why"])
    for c in we[1]: c.font = Font(bold=True); c.fill = HDR
    r = 2
    for n, (text, ents, why) in enumerate(EXAMPLES, start=1):
        toks = text.split()
        tags = ["O"] * len(toks)
        for etext, etype in ents:
            ew = etext.split()
            for i in range(len(toks) - len(ew) + 1):
                if toks[i:i+len(ew)] == ew:
                    tags[i] = f"B-{etype}"
                    for j in range(1, len(ew)): tags[i+j] = f"I-{etype}"
                    break
        for i, (tok, tg) in enumerate(zip(toks, tags)):
            we.cell(row=r, column=1, value=n if i == 0 else None)
            if i == 0:
                c = we.cell(row=r, column=2, value=text)
                c.fill = SENT; c.alignment = Alignment(wrap_text=True, vertical="top")
                c2 = we.cell(row=r, column=5, value=why)
                c2.alignment = Alignment(wrap_text=True, vertical="top")
            we.cell(row=r, column=3, value=tok)
            cc = we.cell(row=r, column=4, value=tg)
            if tg != "O": cc.fill = GOOD; cc.font = Font(bold=True)
            r += 1
        r += 1
    for col, w in (("A",5),("B",46),("C",22),("D",16),("E",52)):
        we.column_dimensions[col].width = w

    ws2 = wb.create_sheet("Tagset")
    ws2.append(["Type", "Use it for"])
    for c in ws2[1]: c.font = Font(bold=True); c.fill = HDR
    for t in TYPES: ws2.append([t, GLOSS[t]])
    ws2.append(["O", "Anything that is not a named entity."])
    ws2.column_dimensions["A"].width = 16
    ws2.column_dimensions["B"].width = 88
    for row in ws2.iter_rows(min_row=2):
        row[1].alignment = Alignment(wrap_text=True, vertical="top")

    ws3 = wb.create_sheet("Annotate")
    ws3.append(["Sent", "Sentence", "Tok", "Word", "Tag", "Note"])
    for c in ws3[1]: c.font = Font(bold=True); c.fill = HDR
    ws3.freeze_panes = "A2"
    rr = 2
    for item in key:
        text = " ".join(item["tokens"])
        for j, tok in enumerate(item["tokens"], start=1):
            ws3.cell(row=rr, column=1, value=item["key_id"])
            if j == 1:
                c = ws3.cell(row=rr, column=2, value=text)
                c.fill = SENT; c.alignment = Alignment(wrap_text=True, vertical="top")
            ws3.cell(row=rr, column=3, value=j)
            ws3.cell(row=rr, column=4, value=tok)
            ws3.cell(row=rr, column=5, value="O")
            rr += 1
    dv = DataValidation(type="list", formula1='"' + ",".join(TAGS) + '"',
                        allow_blank=False, showDropDown=False)
    dv.error = "Choose a tag from the dropdown."; dv.errorTitle = "Invalid tag"
    ws3.add_data_validation(dv); dv.add(f"E2:E{rr-1}")
    for col, w in (("A",6),("B",58),("C",6),("D",22),("E",16),("F",30)):
        ws3.column_dimensions[col].width = w

    wb.save(path)
    return rr - 2

for a in ANNOTATORS:
    p = OUT / f"mizo_ner_annotation_{a}.xlsx"
    n = build(a, p)
    print(f"-> {p.relative_to(ROOT)}   {n:,} token rows, {len(EXAMPLES)} examples")

-> annotation\mizo_ner_annotation_A.xlsx   3,644 token rows, 6 examples
-> annotation\mizo_ner_annotation_B.xlsx   3,644 token rows, 6 examples


## Cell 7: Before you send

In [12]:
tok = sum(len(r["tokens"]) for r in sample)
print("Send each annotator ONE workbook and nothing else:")
for a in ANNOTATORS:
    print(f"   annotation/mizo_ner_annotation_{a}.xlsx")
print("\nDo NOT send annotation/gold_sample_key.json.")
print(f"\nEach annotates {len(sample)} sentences / {tok:,} words "
      f"(roughly {tok/250:.0f} hours).")
print("""
Recommended before the full run:
  1. Both annotators tag the first 25 sentences only.
  2. Compare, and talk through every disagreement.
  3. Agree any convention the guide does not cover, and write it down.
  4. Then continue independently.

This calibration round usually raises agreement substantially, because most
early disagreement is about conventions rather than judgment.
""")

Send each annotator ONE workbook and nothing else:
   annotation/mizo_ner_annotation_A.xlsx
   annotation/mizo_ner_annotation_B.xlsx

Do NOT send annotation/gold_sample_key.json.

Each annotates 300 sentences / 3,644 words (roughly 15 hours).

Recommended before the full run:
  1. Both annotators tag the first 25 sentences only.
  2. Compare, and talk through every disagreement.
  3. Agree any convention the guide does not cover, and write it down.
  4. Then continue independently.

This calibration round usually raises agreement substantially, because most
early disagreement is about conventions rather than judgment.

